# Ensemble Learning: Predictive Maintenance dengan Multiple Models

**Konsep**: Menggabungkan beberapa model (base learners) untuk membuat prediksi yang lebih robust dan akurat.

**Metode Ensemble**:
1. **Voting Classifier**: Hard voting (mayoritas) atau soft voting (rata-rata probabilitas)
2. **Stacking Classifier**: Meta-learner belajar cara menggabungkan prediksi base models

**Base Models**:
- XGBoost (gradient boosting)
- Random Forest (bagging)
- Logistic Regression (linear)
- SGD Classifier (linear, fast)

**⚠️ Penting**: Ensemble methods di scikit-learn **TIDAK** mendukung `partial_fit`. Mereka harus dilatih dengan `fit()` menggunakan semua data sekaligus. Ini berbeda dengan online learning.

**Workflow**:
1. Load data & feature engineering
2. Split data (train/val/test)
3. Train base models individual
4. Train ensemble models (Voting & Stacking)
5. Compare: single best model vs ensemble
6. Evaluate & log ke MLflow

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import (
    VotingClassifier, StackingClassifier, RandomForestClassifier
)
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    precision_score, recall_score, average_precision_score,
    roc_auc_score, ConfusionMatrixDisplay, precision_recall_curve, roc_curve
)
from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ---------------- Configuration ----------------
DATA_SOURCE = os.getenv("DATA_SOURCE", "csv")
DATA_PATH = os.getenv("DATA_PATH", "../data/ai4i2020.csv")

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
MLFLOW_EXPERIMENT = os.getenv("MLFLOW_EXPERIMENT", "ensemble-learning-machine-failure")
REGISTERED_MODEL_NAME = os.getenv("MLFLOW_MODEL_NAME", "machine_failure_ensemble")
MODEL_ALIAS = os.getenv("MLFLOW_MODEL_ALIAS", "champion")

GRID_CV_FOLDS = int(os.getenv("GRID_CV_FOLDS", "3"))

print("✅ Configuration loaded")
print(f"Data: {DATA_PATH}")

## 1. Load Data & Feature Engineering

In [ ]:
COLUMNS = ["udi", "product_id", "type", "air_temp_k", "process_temp_k",
           "rpm", "torque_nm", "tool_wear_min", "machine_failure",
           "twf", "hdf", "pwf", "osf", "rnf"]
INT_COLS = ["udi", "rpm", "tool_wear_min", "machine_failure", "twf", "hdf", "pwf", "osf", "rnf"]
FLOAT_COLS = ["air_temp_k", "process_temp_k", "torque_nm"]

def load_data(path):
    print(f"📂 Loading data from: {path}")
    d = pd.read_csv(path, encoding="utf-8-sig")
    d.columns = COLUMNS
    d = d.astype({**{c: "int64" for c in INT_COLS}, **{c: "float64" for c in FLOAT_COLS}})
    print(f"   Shape: {d.shape}")
    print(f"   Failure rate: {d['machine_failure'].mean():.2%}")
    return d

df = load_data(DATA_PATH)

# Feature engineering
TYPE_MAP = {"L": 0, "M": 1, "H": 2}

def make_features(d: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=d.index)
    out["type_enc"] = d["type"].map(TYPE_MAP)
    out["air_temp_k"] = d["air_temp_k"]
    out["process_temp_k"] = d["process_temp_k"]
    out["rpm"] = d["rpm"]
    out["torque_nm"] = d["torque_nm"]
    out["tool_wear_min"] = d["tool_wear_min"]
    out["temp_diff_k"] = d["process_temp_k"] - d["air_temp_k"]
    out["power_w"] = d["torque_nm"] * d["rpm"] * 2 * np.pi / 60
    out["wear_torque"] = d["tool_wear_min"] * d["torque_nm"]
    return out

X = make_features(df)
y = df["machine_failure"]
FEATURES = list(X.columns)

print(f"\n✅ Features created: {len(FEATURES)} features")
X.head()

## 2. Split Data

Split: 64% train, 16% validation, 20% test

In [ ]:
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.20, stratify=y_tmp, random_state=RANDOM_STATE
)

for name, part in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{name:5s}: {len(part):5d} rows, failure {part.mean():.2%}")

## 3. Define Base Models

Kita akan train 4 base models individual, lalu combine mereka dengan ensemble methods.

In [ ]:
print("\n🔧 Defining Base Models")
print("-" * 50)

# Calculate scale_pos_weight untuk XGBoost (handle imbalanced data)
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

# Base models
base_models = {
    "xgb": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric="aucpr",
        tree_method="hist",
        random_state=RANDOM_STATE,
        use_label_encoder=False
    ),
    "rf": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=5,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "lr": LogisticRegression(
        C=1.0,
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_STATE
    ),
    "sgd": SGDClassifier(
        loss="log_loss",
        alpha=0.0001,
        class_weight="balanced",
        random_state=RANDOM_STATE
    )
}

print(f"✅ Defined {len(base_models)} base models:")
for name, model in base_models.items():
    print(f"   - {name}: {model.__class__.__name__}")

## 4. Train Base Models Individual

Train setiap base model secara terpisah untuk baseline comparison.

In [ ]:
print("\n🚀 Training Base Models Individual")
print("-" * 50)

base_results = {}

for name, model in base_models.items():
    print(f"\n📊 Training {name}...")
    t0 = time.time()
    
    # Train
    if name == "xgb":
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    else:
        model.fit(X_train, y_train)
    
    train_time = time.time() - t0
    
    # Predict on validation set
    y_val_pred = model.predict(X_val)
    y_val_proba = model.predict_proba(X_val)[:, 1]
    
    # Metrics
    val_f1 = f1_score(y_val, y_val_pred)
    val_precision = precision_score(y_val, y_val_pred)
    val_recall = recall_score(y_val, y_val_pred)
    val_pr_auc = average_precision_score(y_val, y_val_proba)
    
    base_results[name] = {
        "model": model,
        "train_time": train_time,
        "val_f1": val_f1,
        "val_precision": val_precision,
        "val_recall": val_recall,
        "val_pr_auc": val_pr_auc
    }
    
    print(f"   ✅ Train time: {train_time:.2f}s")
    print(f"   Val F1: {val_f1:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, PR-AUC: {val_pr_auc:.4f}")

# Find best base model
best_base_name = max(base_results, key=lambda k: base_results[k]["val_f1"])
print(f"\n🏆 Best base model: {best_base_name} (Val F1: {base_results[best_base_name]['val_f1']:.4f})")

## 5. Create Ensemble Models

### 5.1 Voting Classifier (Soft Voting)

Soft voting: rata-rata probabilitas dari semua base models, lalu ambil kelas dengan probabilitas tertinggi.

In [ ]:
print("\n🔗 Creating Ensemble Models")
print("-" * 50)

# Voting Classifier (Soft Voting)
print("\n📊 Creating Voting Classifier (Soft Voting)...")
voting_clf = VotingClassifier(
    estimators=[
        ("xgb", base_models["xgb"]),
        ("rf", base_models["rf"]),
        ("lr", base_models["lr"]),
        ("sgd", base_models["sgd"])
    ],
    voting="soft",  # Soft voting: average probabilities
    n_jobs=-1
)

print("   Training Voting Classifier...")
t0 = time.time()
voting_clf.fit(X_train, y_train)
voting_train_time = time.time() - t0

y_val_pred_voting = voting_clf.predict(X_val)
y_val_proba_voting = voting_clf.predict_proba(X_val)[:, 1]

voting_val_f1 = f1_score(y_val, y_val_pred_voting)
voting_val_pr_auc = average_precision_score(y_val, y_val_proba_voting)

print(f"   ✅ Train time: {voting_train_time:.2f}s")
print(f"   Val F1: {voting_val_f1:.4f}, PR-AUC: {voting_val_pr_auc:.4f}")

### 5.2 Stacking Classifier

Stacking: Meta-learner (Logistic Regression) belajar cara menggabungkan prediksi dari base models.

In [ ]:
print("\n📊 Creating Stacking Classifier...")
stacking_clf = StackingClassifier(
    estimators=[
        ("xgb", base_models["xgb"]),
        ("rf", base_models["rf"]),
        ("lr", base_models["lr"]),
        ("sgd", base_models["sgd"])
    ],
    final_estimator=LogisticRegression(
        C=1.0,
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_STATE
    ),
    cv=3,  # Cross-validation untuk generate meta-features
    stack_method="predict_proba",
    n_jobs=-1
)

print("   Training Stacking Classifier...")
t0 = time.time()
stacking_clf.fit(X_train, y_train)
stacking_train_time = time.time() - t0

y_val_pred_stacking = stacking_clf.predict(X_val)
y_val_proba_stacking = stacking_clf.predict_proba(X_val)[:, 1]

stacking_val_f1 = f1_score(y_val, y_val_pred_stacking)
stacking_val_pr_auc = average_precision_score(y_val, y_val_proba_stacking)

print(f"   ✅ Train time: {stacking_train_time:.2f}s")
print(f"   Val F1: {stacking_val_f1:.4f}, PR-AUC: {stacking_val_pr_auc:.4f}")

## 6. Compare: Single Best Model vs Ensemble

Apakah ensemble benar-benar lebih baik dari single best model?

In [ ]:
print("\n📊 Comparison: Single Best Model vs Ensemble")
print("=" * 70)

comparison_data = {
    "Model": [],
    "Val F1": [],
    "Val PR-AUC": [],
    "Train Time (s)": []
}

# Add base models
for name, result in base_results.items():
    comparison_data["Model"].append(f"Base: {name}")
    comparison_data["Val F1"].append(result["val_f1"])
    comparison_data["Val PR-AUC"].append(result["val_pr_auc"])
    comparison_data["Train Time (s)"].append(result["train_time"])

# Add ensemble models
comparison_data["Model"].append("Ensemble: Voting (Soft)")
comparison_data["Val F1"].append(voting_val_f1)
comparison_data["Val PR-AUC"].append(voting_val_pr_auc)
comparison_data["Train Time (s)"].append(voting_train_time)

comparison_data["Model"].append("Ensemble: Stacking")
comparison_data["Val F1"].append(stacking_val_f1)
comparison_data["Val PR-AUC"].append(stacking_val_pr_auc)
comparison_data["Train Time (s)"].append(stacking_train_time)

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values("Val F1", ascending=False)

print(comparison_df.to_string(index=False))

# Find best model overall
best_model_name = comparison_df.iloc[0]["Model"]
best_model_f1 = comparison_df.iloc[0]["Val F1"]

print(f"\n🏆 Best model overall: {best_model_name} (Val F1: {best_model_f1:.4f})")

# Check if ensemble is better than best base
best_base_f1 = base_results[best_base_name]["val_f1"]
ensemble_improvement = best_model_f1 - best_base_f1

if ensemble_improvement > 0:
    print(f"✅ Ensemble improved F1 by {ensemble_improvement:.4f} over best base model")
else:
    print(f"⚠️  Ensemble did not improve over best base model (change: {ensemble_improvement:.4f})")

## 7. Visualization

In [ ]:
print("\n📈 Visualizing Comparison")
print("-" * 50)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# F1-Score comparison
colors = ["skyblue" if "Base" in m else "lightgreen" for m in comparison_df["Model"]]
axes[0].barh(comparison_df["Model"], comparison_df["Val F1"], color=colors, edgecolor="black")
axes[0].set_xlabel("F1-Score")
axes[0].set_title("Validation F1-Score")
axes[0].grid(alpha=0.3, axis="x")
axes[0].axvline(x=best_base_f1, color="r", linestyle="--", label=f"Best Base: {best_base_f1:.3f}")
axes[0].legend()

# PR-AUC comparison
axes[1].barh(comparison_df["Model"], comparison_df["Val PR-AUC"], color=colors, edgecolor="black")
axes[1].set_xlabel("PR-AUC")
axes[1].set_title("Validation PR-AUC")
axes[1].grid(alpha=0.3, axis="x")

# Train time comparison
axes[2].barh(comparison_df["Model"], comparison_df["Train Time (s)"], color=colors, edgecolor="black")
axes[2].set_xlabel("Train Time (seconds)")
axes[2].set_title("Training Time")
axes[2].grid(alpha=0.3, axis="x")

plt.tight_layout()
plt.show()

print("✅ Visualization complete")

## 8. Evaluate Best Model on Test Set

In [ ]:
print("\n🔬 Evaluating Best Model on Test Set")
print("-" * 50)

# Select best model
if "Voting" in best_model_name:
    best_model = voting_clf
    model_type = "voting"
elif "Stacking" in best_model_name:
    best_model = stacking_clf
    model_type = "stacking"
else:
    base_name = best_model_name.split(": ")[1]
    best_model = base_models[base_name]
    model_type = base_name

# Predict on test set
y_test_pred = best_model.predict(X_test)
y_test_proba = best_model.predict_proba(X_test)[:, 1]

# Metrics
test_f1 = f1_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred)
test_recall = recall_score(y_test, y_test_pred)
test_pr_auc = average_precision_score(y_test, y_test_proba)
test_roc_auc = roc_auc_score(y_test, y_test_proba)

print(f"\n📊 Test Set Performance ({best_model_name}):")
print(f"   F1-Score:  {test_f1:.4f}")
print(f"   Precision: {test_precision:.4f}")
print(f"   Recall:    {test_recall:.4f}")
print(f"   PR-AUC:    {test_pr_auc:.4f}")
print(f"   ROC-AUC:   {test_roc_auc:.4f}")

# Classification report
print("\n📋 Classification Report:")
print(classification_report(y_test, y_test_pred, target_names=["Normal", "Failure"]))

## 9. Confusion Matrix & ROC/PR Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Confusion Matrix
ConfusionMatrixDisplay(confusion_matrix(y_test, y_test_pred),
                       display_labels=["Normal", "Failure"]).plot(cmap="Blues", ax=axes[0])
axes[0].set_title("Confusion Matrix (Test)")

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_test_proba)
axes[1].plot(fpr, tpr, linewidth=2, label=f"ROC (AUC = {test_roc_auc:.3f})")
axes[1].plot([0, 1], [0, 1], "k--", label="Random")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend()
axes[1].grid(alpha=0.3)

# PR Curve
precision, recall, _ = precision_recall_curve(y_test, y_test_proba)
axes[2].plot(recall, precision, linewidth=2, label=f"PR (AP = {test_pr_auc:.3f})")
axes[2].axhline(y=y_test.mean(), color="gray", linestyle="--", label="Baseline")
axes[2].set_xlabel("Recall")
axes[2].set_ylabel("Precision")
axes[2].set_title("Precision-Recall Curve")
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Logging ke MLflow

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)
client = MlflowClient()

print("\n📝 Logging to MLflow")
print("-" * 50)

with mlflow.start_run(run_name=f"ensemble_{model_type}_{time.strftime('%Y%m%d_%H%M%S')}") as run:
    # Tags
    mlflow.set_tags({
        "model_type": model_type,
        "ensemble_method": "voting_soft" if model_type == "voting" else "stacking",
        "data_source": DATA_SOURCE,
        "n_base_models": str(len(base_models))
    })
    
    # Parameters
    mlflow.log_params({
        "ensemble_type": model_type,
        "n_base_models": len(base_models),
        "base_models": list(base_models.keys()),
        "n_features": len(FEATURES),
        "train_samples": len(X_train),
        "val_samples": len(X_val),
        "test_samples": len(X_test)
    })
    
    # Metrics
    mlflow.log_metrics({
        "test_f1": test_f1,
        "test_precision": test_precision,
        "test_recall": test_recall,
        "test_pr_auc": test_pr_auc,
        "test_roc_auc": test_roc_auc,
        "val_f1": best_model_f1,
        "ensemble_improvement": ensemble_improvement
    })
    
    # Log comparison table
    mlflow.log_dict({"comparison": comparison_df.to_dict(orient="records")}, "model_comparison.json")
    
    # Log model metadata
    meta = {
        "features": FEATURES,
        "type_map": TYPE_MAP,
        "ensemble_type": model_type,
        "base_models": list(base_models.keys())
    }
    mlflow.log_dict(meta, "meta/model_meta.json")
    
    # Log model
    sig_X = X_test[:5]
    signature = infer_signature(sig_X, best_model.predict_proba(sig_X)[:, 1])
    
    model_info = mlflow.sklearn.log_model(
        best_model,
        artifact_path="model",
        signature=signature,
        input_example=sig_X
    )
    
    print(f"✅ Run ID: {run.info.run_id}")
    print(f"   Model URI: {model_info.model_uri}")
    
    # Register model
    mv = mlflow.register_model(model_info.model_uri, REGISTERED_MODEL_NAME)
    
    # Set tags
    tags = {
        "features": json.dumps(FEATURES),
        "type_map": json.dumps(TYPE_MAP),
        "ensemble_type": model_type,
        "base_models": json.dumps(list(base_models.keys())),
        "test_f1": f"{test_f1:.4f}",
        "test_pr_auc": f"{test_pr_auc:.4f}",
        "run_id": run.info.run_id
    }
    
    for k, v in tags.items():
        client.set_model_version_tag(REGISTERED_MODEL_NAME, mv.version, k, v)
    
    client.set_registered_model_alias(REGISTERED_MODEL_NAME, MODEL_ALIAS, mv.version)
    
    print(f"\n✅ Model registered: {REGISTERED_MODEL_NAME}@{MODEL_ALIAS} (version {mv.version})")

## 11. Inference Test

In [ ]:
print("\n🔮 Inference Test")
print("-" * 50)

# Load model dari registry
model_uri = f"models:/{REGISTERED_MODEL_NAME}@{MODEL_ALIAS}"
loaded_model = mlflow.sklearn.load_model(model_uri)
mv_loaded = client.get_model_version_by_alias(REGISTERED_MODEL_NAME, MODEL_ALIAS)
feats = json.loads(mv_loaded.tags["features"])

# Test sample
test_sample = {
    "type": "L",
    "air_temp_k": 300.5,
    "process_temp_k": 310.9,
    "rpm": 1500,
    "torque_nm": 55.0,
    "tool_wear_min": 215,
}

test_df = pd.DataFrame([test_sample])
test_X = make_features(test_df)[feats]

proba = float(loaded_model.predict_proba(test_X)[0, 1])
pred = loaded_model.predict(test_X)[0]

print(f"Model: {model_uri} (version {mv_loaded.version})")
print(f"\nInput features:")
print(test_X.T)
print(f"\nFailure probability: {proba:.3f}")
print(f"Prediction: {'FAILURE' if pred == 1 else 'NORMAL'}")
print(f"\n✅ Inference test complete")

## 📝 Summary

**Ensemble Learning**:

✅ **Keuntungan**:
- Lebih robust dan akurat dari single model
- Mengurangi variance (bagging) atau bias (boosting)
- Bisa combine berbagai tipe model (tree-based, linear, dll)

⚠️ **Keterbatasan**:
- **TIDAK support `partial_fit`** (harus `fit()` dengan semua data)
- Lebih lambat training (train multiple models)
- Lebih kompleks untuk deployment
- Tidak cocok untuk online/streaming data

**Voting vs Stacking**:
- **Voting**: Simple, cepat, tidak perlu meta-learner
- **Stacking**: Lebih powerful, meta-learner belajar cara combine, tapi lebih kompleks

**Kapan pakai Ensemble?**
- Data statis (tidak streaming)
- Butuh performa maksimal
- Ada resource komputasi cukup
- Tidak perlu update model frequently

**Kapan pakai Online Learning?**
- Data streaming/real-time
- Butuh update model incremental
- Resource terbatas
- Ada concept drift